## Generate plots
- Eligible datasets:
    - data/STEM-live-data/edx-haadf-beacon_20260308_180152
    - data/STEM-live-data/edx-haadf-ei_20260308_181733
    - data/STEM-live-data/edx-haadf-MU_20260308_183249


In [ ]:
## load data
# read files
import pickle
import numpy as np

def read_pickle_al(file_path: str) -> dict:
    with open(file_path, "rb") as f:
        pkl_data_al = pickle.load(f)
    return pkl_data_al




## 1. Haadf with spectra

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re

# ── load ──────────────────────────────────────────────────────────────────────
path_to_haadf_tiff = "edx-haadf-beacon_20260308_180152/HAADF_image_20260308_180153.tiff"
pixel_size_m = 1.45e-9

haadf = tifffile.imread(path_to_haadf_tiff).squeeze()
h, w = haadf.shape


# ── scalebar ──────────────────────────────────────────────────────────────────
def add_scalebar(ax, img_w, img_h, pixel_size_m, color="white", fontsize=9):
    fov = img_w * pixel_size_m
    candidates = [v * 10**e for e in range(-12, -5) for v in (1, 2, 5)]
    bar_m = min(candidates, key=lambda x: abs(x / fov - 0.2))
    bar_px = bar_m / pixel_size_m

    if bar_m < 1e-9:
        bar_lbl = f"{bar_m * 1e12:.4g} pm"
    elif bar_m < 1e-6:
        bar_lbl = f"{bar_m * 1e9:.4g} nm"
    else:
        bar_lbl = f"{bar_m * 1e6:.4g} µm"

    bx = img_w * 0.96 - bar_px
    by = img_h * 0.93
    bh = img_h * 0.012

    ax.add_patch(patches.Rectangle((bx, by), bar_px, bh, color=color, zorder=5))
    ax.text(
        bx + bar_px / 2,
        by - bh * 0.8,
        bar_lbl,
        color=color,
        fontsize=fontsize,
        ha="center",
        va="bottom",
        fontweight="bold",
        zorder=6,
    )

# ── plot only HAADF ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "sans-serif",
    "figure.facecolor": "white"
})

fig, ax = plt.subplots(figsize=(5, 5 * h / w))

ax.imshow(
    haadf,
    cmap="inferno",
    vmin=np.percentile(haadf, 0.5),
    vmax=np.percentile(haadf, 99.5),
)
ax.axis("off")

add_scalebar(ax, w, h, pixel_size_m)


plt.tight_layout()
plt.show()

## 2. Active learning trajectory in real space

In [ ]:
file_path_al_ei_pfm = "edx-haadf-ei_20260308_181733/Dataset_seed5_live_mic_BO_10_epochs100_budget_100_sum_ws5_20260308_181733/Active_learning_statistics.pkl"
file_path_al_mu_pfm = "edx-haadf-MU_20260308_183249/Dataset_seed5_live_mic_BO_10_epochs100_budget_100_sum_ws5_20260308_183249/Active_learning_statistics.pkl"
file_path_al_bn_pfm = "edx-haadf-beacon_20260308_180152/Dataset_seed5_live_mic_BO_10_epochs100_budget_100_sum_ws5_20260308_180152/Active_learning_statistics.pkl"

pkl_data_al_ei = read_pickle_al(file_path=file_path_al_ei_pfm)
pkl_data_al_mu = read_pickle_al(file_path=file_path_al_mu_pfm)
pkl_data_al_bn = read_pickle_al(file_path=file_path_al_bn_pfm)

In [ ]:
def get_acq_ccords_seed_indices_and_t(pkl_data_al: dict):
    indices_all = np.asarray(pkl_data_al["indices_all"])          # (N, 2)
    acq_idx = np.asarray(pkl_data_al["acquired_order"], dtype=int)  # (T,)
    acq_coords = indices_all[acq_idx]                             # (T, 2)
    t = np.arange(len(acq_coords))
    
    # Seeds
    seed_indices = np.asarray(pkl_data_al["seed_indices"], dtype=int)
    seed_coords = indices_all[seed_indices]                       # (S, 2)
    
    # All patches (aligned with indices_all)
    patches_all = np.asarray(pkl_data_al["features"])             # (N, 16, 16)

    # Extract patches
    seed_patches = patches_all[seed_indices]                      # (S, 16, 16)
    acq_patches = patches_all[acq_idx]                             # (T, 16, 16)

    return t, acq_coords, seed_coords, acq_idx, seed_indices, patches_all, seed_patches, acq_patches

In [ ]:
# prep results for 3 cases

## EI
t_ei, acq_coords_ei, seed_coords_ei, acq_idx_order_ei, seed_indices_ei, patches_all, seed_patches_ei, acq_patches_ei = get_acq_ccords_seed_indices_and_t(pkl_data_al_ei)


## UCB-MU
t_mu, acq_coords_mu, seed_coords_mu, acq_idx_order_mu, seed_indices_mu, patches_all, seed_patches_mu, acq_patches_mu  = get_acq_ccords_seed_indices_and_t(pkl_data_al_mu)


## Beacon
t_bn, acq_coords_bn, seed_coords_bn, acq_idx_order_bn, seed_indices_bn, patches_all, seed_patches_bn, acq_patches_bn   = get_acq_ccords_seed_indices_and_t(pkl_data_al_bn)


In [ ]:
# ── helper: scalebar ──────────────────────────────────────────────────────────
def add_scalebar(ax, pixel_size_nm, color="white", fontsize=7,
                 bar_nm_fixed=None, fraction=0.2):
    h, w = ax.get_images()[0].get_array().shape[:2]
    bar_nm = (bar_nm_fixed if bar_nm_fixed is not None
              else min([v * 10**e for e in range(-1, 6) for v in (1, 2, 5)],
                       key=lambda x: abs(x / (w * fraction * pixel_size_nm) - 1)))
    bar_px = bar_nm / pixel_size_nm
    bx = w * 0.96 - bar_px
    by, bh = h * 0.93, h * 0.012
    ax.add_patch(patches.Rectangle((bx, by), bar_px, bh, color=color, zorder=5))
    label = f"{bar_nm:.4g} nm" if bar_nm < 1000 else f"{bar_nm/1000:.4g} µm"
    ax.text(bx + bar_px / 2, by - bh * 0.5, label,
            color=color, fontsize=fontsize, ha="center", va="bottom",
            fontfamily="monospace", fontweight="bold", zorder=5)

# ── figure geometry ───────────────────────────────────────────────────────────
panel_size  = 1.8
cbar_w      = 0.20
hgap        = 0.15                                        # tighter, no per-panel cbars
lm, rm      = 0.15, 0.45                                  # extra right margin for shared cbar
tm, bm      = 0.25, 0.15

# columns: img | gap | img | gap | img | gap | shared_cbar
col_w = [panel_size, hgap, panel_size, hgap, panel_size, hgap, cbar_w]

fig_w = lm + sum(col_w) + rm
fig_h = tm + panel_size + bm

fig = plt.figure(figsize=(fig_w, fig_h))

gs = gridspec.GridSpec(
    1, 7,
    figure=fig,
    width_ratios=col_w,
    left   = lm / fig_w,
    right  = 1 - rm / fig_w,
    top    = 1 - tm / fig_h,
    bottom = bm / fig_h,
    wspace = 0.12,
)

ax_ei  = fig.add_subplot(gs[0, 0])
ax_mu  = fig.add_subplot(gs[0, 2])
ax_bn  = fig.add_subplot(gs[0, 4])
cax    = fig.add_subplot(gs[0, 6])                        # single shared colorbar

# ── scatter panels ────────────────────────────────────────────────────────────
traj_panels = [
    (ax_ei, haadf, ei_x, ei_y, t_ei, seed_x_ei, seed_y_ei),
    (ax_mu, haadf, mu_x, mu_y, t_mu, seed_x_mu, seed_y_mu),
    (ax_bn, haadf, bn_x, bn_y, t_bn, seed_x_bn, seed_y_bn),
]

for ax, bg, sx, sy, t, sdx, sdy in traj_panels:
    ax.imshow(bg, cmap="gray", interpolation="nearest")
    ax.axis("off")
    add_scalebar(ax, 1.45)

    ax.scatter(sdx, sdy, s=18, c="white",
               edgecolors="black", linewidths=0.6, marker="o", zorder=3)

    sc = ax.scatter(sx, sy, c=t, cmap="plasma", vmin=0, vmax=t_max,
                    s=14, marker="o", edgecolors="k", linewidths=0.3, zorder=4)

# ── single shared colorbar ────────────────────────────────────────────────────
cb = fig.colorbar(sc, cax=cax)
cb.set_label("Acquisition step", fontsize=7, fontweight="bold", labelpad=4)
cb.ax.tick_params(labelsize=6, width=0.5)
cb.outline.set_linewidth(0.5)
ticks = np.linspace(0, t_max, 5)
cb.set_ticks(ticks)
cb.set_ticklabels([f"{v:.0f}" for v in ticks])

# ── panel labels ──────────────────────────────────────────────────────────────

for ax, letter, name in zip([ax_ei, ax_mu, ax_bn], ["a", "b", "c"], ["EI", "MU", "BN"]):
    ax.set_title(name, fontsize=7, fontweight="bold", color="black", pad=3)
    ax.text(0.02, 0.98, letter, transform=ax.transAxes,
            fontsize=9, fontweight="bold", color="white", va="top", ha="left")

# fig.savefig("fig3.pdf", dpi=600, bbox_inches="tight")
# fig.savefig("fig3.png", dpi=600, bbox_inches="tight")
plt.show()

## 3. Surrogate mean and uncertainity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── consistent palette ────────────────────────────────────────────────────────
colors = {"EI": "#1f77b4", "MU": "#e377c2", "Beacon": "#ff7f0e"}
labels_display = {"EI": "EI", "MU": "UCB-MU", "Beacon": "Beacon"}

data_sets = [
    ("EI",     t_ei, pkl_data_al_ei, acq_idx_order_ei),
    ("MU",     t_mu, pkl_data_al_mu, acq_idx_order_mu),
    ("Beacon", t_bn, pkl_data_al_bn, acq_idx_order_bn),
]

panels = [
    {
        "ylabel": "Surrogate mean",
        "getter": lambda name, t, pkl, acq: (t, pkl["mean_y_pred_mean_al"]),
    },
    {
        "ylabel": "Surrogate uncertainty",
        "getter": lambda name, t, pkl, acq: (t, pkl["mean_y_pred_variance_al"]),
    },
]

# ── style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 7,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

panel_w, panel_h = 1.9, 1.6
hgap, lm, rm, tm, bm = 0.65, 0.55, 0.15, 0.20, 0.45

fig_w = lm + 2 * panel_w + hgap + rm
fig_h = tm + panel_h + bm

fig = plt.figure(figsize=(fig_w, fig_h))
gs = gridspec.GridSpec(1, 3, figure=fig,
    width_ratios=[panel_w, hgap, panel_w],
    left=lm/fig_w, right=1-rm/fig_w,
    top=1-tm/fig_h, bottom=bm/fig_h,
    wspace=0)

axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 2])]

# ── plot ──────────────────────────────────────────────────────────────────────
for ax, panel, letter in zip(axes, panels, ["a", "b"]):
    for name, t, pkl, acq_idx in data_sets:
        x, y = panel["getter"](name, t, pkl, acq_idx)
        ax.plot(x, y, linewidth=1.6, color=colors[name], alpha=0.92, label=labels_display[name])
        ax.scatter(x[-1], y[-1], color=colors[name], s=22,
                   edgecolors="k", linewidths=0.4, zorder=5)

    ax.set_ylabel(panel["ylabel"], fontsize=7, fontweight="bold", labelpad=4)
    ax.set_xlabel("Acquisition step", fontsize=7, labelpad=3)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=6, width=0.5)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.5)
    ax.text(0.03, 0.97, letter, transform=ax.transAxes,
            fontsize=9, fontweight="bold", color="black", va="top", ha="left")

axes[0].legend(loc="best", frameon=True, framealpha=0.9,
               edgecolor="lightgray", fontsize=6, handlelength=2.0, borderpad=0.3)

# fig.savefig("fig4.pdf", dpi=400, bbox_inches="tight")
# fig.savefig("fig4.png", dpi=400, bbox_inches="tight")
plt.show()

## 4. Patch space coverage

In [ ]:
import numpy as np

def _kmeans_numpy(X: np.ndarray, k: int, n_iters: int = 25, seed: int = 0):
    """
    Simple numpy k-means.
    X: (N, D) float
    Returns:
      labels: (N,) int in [0, k-1]
      centers: (k, D)
    """
    rng = np.random.default_rng(seed)
    N, D = X.shape

    # init centers from random points
    centers = X[rng.choice(N, size=k, replace=False)].copy()

    for _ in range(n_iters):
        # assign
        # dist^2 = ||x||^2 + ||c||^2 - 2 x·c
        x2 = np.sum(X * X, axis=1, keepdims=True)            # (N,1)
        c2 = np.sum(centers * centers, axis=1)[None, :]      # (1,k)
        d2 = x2 + c2 - 2.0 * (X @ centers.T)                 # (N,k)
        labels = np.argmin(d2, axis=1)                       # (N,)

        # update
        new_centers = centers.copy()
        for j in range(k):
            mask = (labels == j)
            if np.any(mask):
                new_centers[j] = X[mask].mean(axis=0)
            else:
                # empty cluster -> re-seed
                new_centers[j] = X[rng.integers(0, N)]
        centers = new_centers

    return labels, centers


def compute_patch_space_coverage(acq_idx, patches_all, k=80, embed_dim=32, seed=0, n_kmeans_iters=25):
    """
    Patch-space discovery curve (coverage vs step).
    - Clusters ALL patches once (discrete patch diversity).
    - Coverage(t) = fraction of clusters discovered by acquired patches up to t.

    acq_idx: (T,) indices into patches_all
    patches_all: (N, 16, 16) or (N, H, W[, C])
    """
    acq_idx = np.asarray(acq_idx, dtype=int)
    P = np.asarray(patches_all)

    # flatten patches -> (N, D)
    X = P.reshape(P.shape[0], -1).astype(np.float32)

    # normalize (important so intensity scale doesn't dominate)
    X = X - X.mean(axis=1, keepdims=True)
    denom = X.std(axis=1, keepdims=True) + 1e-8
    X = X / denom

    # cheap embedding to speed kmeans (random projection)
    rng = np.random.default_rng(seed)
    W = rng.normal(size=(X.shape[1], embed_dim)).astype(np.float32)
    Z = X @ W  # (N, embed_dim)

    # cluster all patches
    labels, _ = _kmeans_numpy(Z, k=k, n_iters=n_kmeans_iters, seed=seed)

    # which clusters exist at all?
    required_clusters = set(np.unique(labels).tolist())

    # step through acquisitions
    coverage = []
    seen = set()

    for idx in acq_idx:
        seen.add(int(labels[idx]))
        coverage.append(len(seen) / len(required_clusters))

    return np.asarray(coverage, dtype=float)

In [ ]:
patches_all = np.asarray(pkl_data_al_bn["features"])

cover_ei_patch = compute_patch_space_coverage(acq_idx_order_ei, patches_all, k=100)
cover_mu_patch = compute_patch_space_coverage(acq_idx_order_mu, patches_all, k=100)
cover_bn_patch = compute_patch_space_coverage(acq_idx_order_bn, patches_all, k=100)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

for name, y, color in [("EI", cover_ei_patch, colors["EI"]),
                        ("MU", cover_mu_patch, colors["MU"]),
                        ("Beacon", cover_bn_patch, colors["Beacon"])]:
    x = np.arange(len(y))
    ax.plot(x, y, linewidth=1.6, color=color, alpha=0.92, label=labels_display[name])
    ax.scatter(x[-1], y[-1], color=color, s=22, edgecolors="k", linewidths=0.4, zorder=5)

ax.set_ylabel("Patch space coverage", fontsize=7, fontweight="bold", labelpad=4)
ax.set_xlabel("Acquisition step", fontsize=7, labelpad=3)
ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=6, width=0.5)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(0.5)
ax.legend(loc="lower right", frameon=True, framealpha=0.9,
          edgecolor="lightgray", fontsize=6, handlelength=2.0, borderpad=0.3)
ax.text(0.03, 0.97, "a", transform=ax.transAxes,
        fontsize=9, fontweight="bold", color="black", va="top", ha="left")

plt.tight_layout()
# plt.savefig("patch_coverage.pdf", dpi=400, bbox_inches="tight")
# plt.savefig("patch_coverage.png", dpi=400, bbox_inches="tight")
plt.show()

## 5. Timning plot - ignore seed points
- Hardware time
- model training time
- ei time
- bn time
- mu time

In [ ]:
## load data
# read files
import pickle
import numpy as np

def read_pickle_al(file_path: str) -> dict:
    with open(file_path, "rb") as f:
        pkl_data_al = pickle.load(f)

    # # print keys
    # if isinstance(pkl_data_al, dict):
    #     print("Keys in pickle file:")
    #     for key in pkl_data_al.keys():
    #         print(f" - {key}")
    # else:
    #     print("Loaded object is not a dictionary. Type:", type(pkl_data_al))

    return pkl_data_al

In [ ]:
file_path_al_ei_pfm = "edx-haadf-ei_20260308_181733/Dataset_seed5_live_mic_BO_10_epochs100_budget_100_sum_ws5_20260308_181733/Active_learning_statistics.pkl"
file_path_al_mu_pfm = "edx-haadf-MU_20260308_183249/Dataset_seed5_live_mic_BO_10_epochs100_budget_100_sum_ws5_20260308_183249/Active_learning_statistics.pkl"
file_path_al_bn_pfm = "edx-haadf-beacon_20260308_180152/Dataset_seed5_live_mic_BO_10_epochs100_budget_100_sum_ws5_20260308_180152/Active_learning_statistics.pkl"

pkl_data_al_ei = read_pickle_al(file_path=file_path_al_ei_pfm)
pkl_data_al_mu = read_pickle_al(file_path=file_path_al_mu_pfm)
pkl_data_al_bn = read_pickle_al(file_path=file_path_al_bn_pfm)

In [ ]:
def get_timing_info(pkl_data_al: dict):
    time_hw = pkl_data_al.get("time_hw")
    time_gp_train = pkl_data_al.get("time_gp_train")
    time_acq_opt = pkl_data_al.get("time_acq_opt")

    return time_hw, time_gp_train, time_acq_opt

In [ ]:
# prep results for 3 cases

## EI
time_hw_ei, time_gp_train_ei, time_acq_opt_ei = get_timing_info(pkl_data_al_ei)

## UCB-MU
time_hw_mu, time_gp_train_mu, time_acq_opt_mu = get_timing_info(pkl_data_al_mu)

## Beacon
time_hw_bn, time_gp_train_bn, time_acq_opt_bn = get_timing_info(pkl_data_al_bn)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(4.5, 1.8))

# ── left: hardware + GP training ─────────────────────────────────────────────
ax1.plot(time_hw_bn[1:],       linewidth=0.9, color="gray",  alpha=0.85, label="Hardware call")
ax1.plot(time_gp_train_bn[1:], linewidth=0.9, color="black", alpha=0.85, label="Model training")
ax1.set_ylabel("Time (s)", fontsize=7, fontweight="bold", labelpad=4)
ax1.legend(loc="center right", frameon=True, framealpha=0.9,
           edgecolor="lightgray", fontsize=6, handlelength=2.0)

# ── right: acquisition functions ─────────────────────────────────────────────
ax2.plot(time_acq_opt_ei[1:],  linewidth=0.9, color=colors["EI"],     alpha=0.85, label="EI")
ax2.plot(time_acq_opt_mu[1:],  linewidth=0.9, color=colors["MU"],     alpha=0.85, label="UCB-MU")
ax2.plot(time_acq_opt_bn[1:],  linewidth=0.9, color=colors["Beacon"], alpha=0.85, label="Beacon")
ax2.legend(loc="upper right", frameon=True, framealpha=0.9,
           edgecolor="lightgray", fontsize=6, handlelength=2.0)

for ax, letter in zip([ax1, ax2], ["a", "b"]):
    ax.set_xlabel("Acquisition step", fontsize=7, labelpad=3)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=6, width=0.5)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.5)
    ax.text(0.03, 0.97, letter, transform=ax.transAxes,
            fontsize=9, fontweight="bold", color="black", va="top", ha="left")

plt.tight_layout(pad=0.5)
plt.savefig("fig_timing.pdf", dpi=600, bbox_inches="tight")
plt.savefig("fig_timing.png", dpi=600, bbox_inches="tight")
plt.show()